# Strut-and-Tie by Topology Optimization

Go from a **generic shape** (a rectangle or an arbitrary polygon) to an
*optimized* strut-and-tie truss, automatically. This complements the manual
`StrutAndTieModel` workflow in *Strut and Tie Demo.ipynb*: there you draw the
truss; here civilpy **discovers** it.

Pipeline (see `docs/StrutAndTieSolver.md`): mesh the region → SIMP topology
optimization → skeletonize the density into a truss → solve (direct stiffness)
→ size ties and check nodes (AASHTO 5.8.2) → cost.

## Workflow 1 — manual: define the region in Python

A simply supported deep beam, 20 ft × 10 ft, 2 ft thick, with a 600 kip load at
midspan top. Supports near the bottom corners. *Geometry carries what is
spatial; tags carry what is scalar* — the same principle as the Rhino bridge.

In [ ]:
from civilpy.structural.stm_topology import DRegionProblem, Material

p = DRegionProblem.rectangle(20, 10, thickness=2.0,
                             material=Material(f_c=5.0), vol_frac=0.35)
p.add_support(1, 0, fix_x=True, fix_y=True, bearing=1.5)   # pin
p.add_support(19, 0, fix_x=False, fix_y=True, bearing=1.5)  # roller
p.add_load(10, 10, fy=-600, bearing=1.5)

result = p.solve(nelx=100)   # mesh → SIMP → extract → solve → size → cost
print(result.summary())

In [ ]:
# The SIMP density (grey) with the extracted strut-and-tie truss overlaid:
# blue dashed = struts (compression), red = ties (tension).
result.plot();

In [ ]:
# result.model is an ordinary StrutAndTieModel — solved forces feed the AASHTO
# capacity checks unchanged.
for member, force in result.forces.items():
    kind = 'tie' if force > 0 else 'strut'
    print(f'{member}: {force:+8.1f} kip ({kind})')

In [ ]:
# Tie reinforcement (AASHTO 5.8.2.4) and the ODOT cost take-off.
for t in result.report.ties:
    print(f'{t.member}: {t.force:6.1f} kip → {t.bar_count} x #{t.bar_size} '
          f'(As={t.a_st_provided:.2f} in^2, ratio={t.check.ratio:.2f})')
c = result.report.cost
print(f'\nTake-off: {c.concrete_cy:.1f} cy concrete + {c.steel_lb:.0f} lb steel '
      f'= ${c.total:,.0f}')

## Arbitrary polygons, voids, and keep-in solids

The optimizer takes any closed polygon, plus optional `voids` (mandatory holes)
and `solids` (keep-in zones). Here a duct void forces the load path around it.

In [ ]:
pv = DRegionProblem.rectangle(20, 10, thickness=2.0, vol_frac=0.4)
pv.voids = [[(8, 4), (12, 4), (12, 6), (8, 6)]]   # a duct opening
pv.add_support(1, 0, bearing=1.5)
pv.add_support(19, 0, fix_x=False, bearing=1.5)
pv.add_load(10, 10, fy=-600, bearing=1.5)
rv = pv.solve(nelx=100)
print(rv.summary())
rv.plot();

## Workflow 2 — Rhino: draw the region, optimize here

Instead of `add_*` calls, draw the concrete outline in Rhino as a closed curve
tagged `stm.kind=region` (with `stm.thickness`, `stm.fc`), tag the supports and
loads exactly as in the drawn-truss workflow, and read it in one call. The tag
schema is the frozen contract in `docs/Rhino Design Philosophy.md`. Needs the
optional `rhino3dm` dependency (`pip install civilpy[rhino]`).

In [ ]:
# from civilpy.structural.stm_topology import DRegionProblem
#
# problem = DRegionProblem.from_3dm('PierCap_Region.3dm')  # stm.kind=region + supports/loads
# result = problem.solve()
# result.plot()
#
# # round-trip the discovered truss back to Rhino for review
# from civilpy.structural import rhino_stm
# rhino_stm.results_to_3dm(result.model, 'PierCap_STM_results.3dm')

## Under the hood

Each stage is independently usable:

```python
from civilpy.structural.stm_topology import GroundMesh, optimize_density, extract_truss
mesh    = GroundMesh(p, nelx=120)            # Phase 1
density = optimize_density(mesh, vol_frac=0.35)   # Phase 2 (SIMP)
model   = extract_truss(density)             # Phase 3 (skeleton → truss)
forces  = model.solve_fully_stressed()       # Phase 4 (direct stiffness)
```

The indeterminate solver also stands alone on any `StrutAndTieModel` via
`solve(method='auto')`.